pdf is a claude provided skill runs on server side

In [ ]:
from pathlib import Path

from util.claude_client import  get_claude_client_and_default_model

client , llm = get_claude_client_and_default_model('claude')


In [ ]:
message = client.messages.create(
    model = llm ,
    max_tokens = 16000,
    container={
        "skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]
    },
    messages = [
        {
            'role' : 'user',
            "content": "Generate a PDF invoice template"

        }

    ],
    tools=[{"type": "code_execution_20260521", "name": "code_execution"}]
)

file_id = None
for block in message.content:
    if block.type == "bash_code_execution_tool_result":
        if block.content.type == "bash_code_execution_result":
            for output in block.content.content:
                file_id = output.file_id


if file_id:
    # Download the file and save it
    output_path = Path.cwd() / "invoice_template.pdf"
    file_content = client.files.download(file_id=file_id)
    file_content.write_to_file(output_path)
    print(f"invoice template saved to {output_path}")
else:
    raise RuntimeError(
        f"Claude returned no downloadable file (stop_reason={message.stop_reason}). "
        "Inspect message.content for tool errors or an unfinished generation."
    )